In [0]:

dbutils.widgets.text("batch_id", "")
dbutils.widgets.text("run_date", "")
dbutils.widgets.dropdown(
    "load_type", "FULL", ["FULL", "INCREMENTAL"]
)
dbutils.widgets.text("manifest_path", "")


In [0]:
%run ./common

In [0]:
# Process existing raw file manifest. INCREMENTAL API ingestion runs beforehand.
import json, uuid, traceback
from pathlib import Path
from datetime import date

spark.conf.set('spark.sql.session.timeZone','UTC')
for key in ('batch_id','run_date'):
    dbutils.widgets.text(key,'')
dbutils.widgets.dropdown('load_type','FULL',['FULL','INCREMENTAL'])
dbutils.widgets.text('manifest_path','')
batch_id = dbutils.widgets.get('batch_id').strip()
run_date = dbutils.widgets.get('run_date').strip()
load_type = dbutils.widgets.get('load_type').strip()
manifest_override = dbutils.widgets.get('manifest_path').strip()
start=now(); execution_id=str(uuid.uuid4()); processed=0; errors=None
try:
    if not batch_id: raise ValueError('batch_id required; use the raw full-load batch id')
    date.fromisoformat(run_date)
    if load_type not in ('FULL','INCREMENTAL'): raise ValueError('Invalid load_type')
    manifests = Path(ROOT)/'_manifests'
    if manifest_override:
        manifest_file=Path(manifest_override)
    elif load_type=='INCREMENTAL':
        manifest_file=manifests/f'incremental_{batch_id}.json'
    else:
        matches=list(manifests.glob(f'full_load_*_{batch_id}.json'))
        if len(matches)!=1: raise ValueError('FULL manifest not unique/found; supply manifest_path')
        manifest_file=matches[0]
    manifest=json.loads(manifest_file.read_text(encoding='utf-8'))
    if manifest.get('batch_id')!=batch_id: raise ValueError('Manifest batch ID mismatch')
    files=[x for x in manifest['files'] if x['entity'] in ENTITIES and x['repository'] in REPOS]
    keys=[(x['repository'],x['entity']) for x in files]
    if len(files)!=9 or len(set(keys))!=9:
        raise ValueError('Manifest must contain exactly one file for each of 9 repo/entity pairs')
    params=[]
    for item in files:
        if not Path(item['path']).is_file(): raise FileNotFoundError(item['path'])
        params.append({'batch_id':batch_id,'run_date':run_date,'load_type':load_type,
                       'entity_type':item['entity'],'file_path':item['path'],
                       'repository':item['repository']})

    # All Bronze batches finish before any Silver processing.
    for args in params:
        output=json.loads(dbutils.notebook.run('./raw_to_bronze',0,args))
        if output.get('status')!='SUCCESS': raise RuntimeError('Bronze did not succeed: '+str(output))
        processed+=int(output['rows_processed'])
        print('BRONZE',args['repository'],args['entity_type'],output)
    for args in params:
        output=json.loads(dbutils.notebook.run('./bronze_to_silver',0,args))
        if output.get('status')!='SUCCESS': raise RuntimeError('Silver did not succeed: '+str(output))
        processed+=int(output['rows_processed'])
        print('SILVER',args['repository'],args['entity_type'],output)
    log(spark,execution_id,batch_id,'driver_notebook','PIPELINE',run_date,
        load_type,start,'SUCCESS',processed)
except Exception:
    errors=traceback.format_exc()
    try: log(spark,execution_id,batch_id,'driver_notebook','PIPELINE',run_date,
             load_type,start,'FAILED',processed,errors)
    except Exception as ex: print('AUDIT LOG ERROR:',ex)
    raise
# Execute only outside exception handler (do not mask failures).
dbutils.notebook.exit(json.dumps({'status':'SUCCESS','batch_id':batch_id,
                                  'rows_processed':processed}))
